# Stage 3 -- Feature Naming: Daily Means & Daily Full Moments

## Purpose
Two short utility notebooks that build human-readable factor inventory CSVs for the two daily model-ready tables. Each notebook reads the parquet schema to get the exact column names present in the final table, then cross-references the Stage 1.5 factor inventories to attach source, category, and description metadata to every feature. The outputs are reference CSVs used for model interpretation, feature selection, and documentation.

---

## Notebook 01: Daily Means Factor Inventory
**Input:** `Data/Data_Collection/Final/Stage_3_Model_Ready/model_market_daily_means.parquet`
**Inventories:** `stock_daily_factor_inventory_final.csv`, `macro_daily_factor_inventory_final.csv`

### Logic
Column names are read directly from the parquet schema (no data loaded). The target (`target_daily_return`) and `date` are excluded, leaving the feature columns. Each feature is looked up in one of two dictionaries built from the Stage 1.5 inventories:

- **Stock inventory lookup:** stock daily factors were aggregated as cwmean with column names unchanged. One naming conflict from the Stage 2 merge required renaming: `skew_chg_5d` → `stock_skew_chg_5d`. The renamed column is added to the lookup map pointing to the original entry.
- **Macro inventory lookup:** Panel C macro daily factors pass through with no renaming.

Unmatched columns (neither in stock nor macro inventory) are flagged and listed.

### Output Columns
`column`, `panel` (A stock daily / C macro daily), `agg_method` (cwmean / raw level), `source`, `category`, `description`

**Output:** `Data/Data_Collection/Final/Stage_3_Model_Ready/daily_means_factor_inventory.csv`

---


In [1]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

# Load the 398 feature names from the daily means table
BASE = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready')
schema = pq.read_schema(BASE / 'model_market_daily_means.parquet')
all_cols = [f.name for f in schema]
features = [c for c in all_cols if c not in ['date', 'target_daily_return']]

# Load both inventories
INV_DIR = Path('../../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering')
stock_inv = pd.read_csv(INV_DIR / 'stock_daily_factor_inventory_final.csv')
macro_inv = pd.read_csv(INV_DIR / 'macro_daily_factor_inventory_final.csv')

# The stock factors were aggregated as cwmean — column names unchanged
# But one conflict was renamed: skew_chg_5d → stock_skew_chg_5d
stock_inv_map = stock_inv.set_index('column')[['source', 'category', 'description']].to_dict('index')
macro_inv_map = macro_inv.set_index('column')[['source', 'category', 'description']].to_dict('index')

# Also check for the renamed conflict
stock_inv_map['stock_skew_chg_5d'] = stock_inv_map.get('skew_chg_5d', {})

# Build the merged inventory
rows = []
matched = 0
unmatched = []

for f in features:
    if f in stock_inv_map:
        info = stock_inv_map[f]
        rows.append({
            'column': f,
            'panel': 'A (stock daily)',
            'agg_method': 'cwmean',
            'source': info.get('source', ''),
            'category': info.get('category', ''),
            'description': info.get('description', ''),
        })
        matched += 1
    elif f in macro_inv_map:
        info = macro_inv_map[f]
        rows.append({
            'column': f,
            'panel': 'C (macro daily)',
            'agg_method': 'raw level',
            'source': info.get('source', ''),
            'category': info.get('category', ''),
            'description': info.get('description', ''),
        })
        matched += 1
    else:
        rows.append({
            'column': f,
            'panel': '???',
            'agg_method': '',
            'source': '',
            'category': '',
            'description': '',
        })
        unmatched.append(f)

result = pd.DataFrame(rows)

print(f"Total features: {len(features)}")
print(f"Matched: {matched}")
print(f"Unmatched: {len(unmatched)}")
if unmatched:
    print(f"\nUnmatched columns:")
    for c in unmatched:
        print(f"  {c}")

# Save
out_path = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready/daily_means_factor_inventory.csv')
result.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(f"  {len(result)} rows")
result.head(10)

Total features: 398
Matched: 398
Unmatched: 0

Saved: ..\..\..\..\Data\Data_Collection\Final\Stage_3_Model_Ready\daily_means_factor_inventory.csv
  398 rows


,column,panel,agg_method,source,category,description
0,dlyretx,A (stock daily),cwmean,CRSP,return,Daily return excluding dividends (price return...
1,dlyreti,A (stock daily),cwmean,CRSP,return,Daily return from dividends only (income return)
2,bid_ask_spread,A (stock daily),cwmean,CRSP,liquidity,Normalised bid-ask spread: |ask - bid| / midpoint
3,bs_ratio_inst50k_num,A (stock daily),cwmean,TAQ,order_flow,"Buy-sell ratio by trade count, institutional >..."
4,bs_ratio_inst50k_vol,A (stock daily),cwmean,TAQ,order_flow,"Buy-sell ratio by volume, institutional >$50K"
5,bs_ratio_num,A (stock daily),cwmean,TAQ,order_flow,"Buy-sell ratio by trade count, all trades"
6,bs_ratio_retail_num,A (stock daily),cwmean,TAQ,order_flow,"Buy-sell ratio by trade count, retail"
7,bs_ratio_retail_vol,A (stock daily),cwmean,TAQ,order_flow,"Buy-sell ratio by volume, retail"
8,bs_ratio_vol,A (stock daily),cwmean,TAQ,order_flow,"Buy-sell ratio by volume, all trades"
9,dollarpriceimpact_lr_ave,A (stock daily),cwmean,TAQ,price_impact,"Dollar price impact, Lee-Ready, equal-weighted..."
